<a href="https://colab.research.google.com/github/rsher60/LLM_Codebase/blob/main/finetuning_udemy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -U transformers datasets seaborn bertviz umap-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 78.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.5/157.5 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.2/89.2 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
from datasets import load_dataset

dataset = load_dataset('dair-ai/emotion')

README.md: 0.00B [00:00, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [4]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})

In [5]:
df = dataset['train'].to_pandas()

In [6]:
df.head()

,text,label
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,3
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,3


In [7]:
label_names = dataset['train'].features

In [8]:
label_names['label']

ClassLabel(names=['sadness', 'joy', 'love', 'anger', 'fear', 'surprise'], id=None)

In [9]:
a = zip([0,1,2,3,4,5], ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise'])

In [10]:
df.dtypes

,0
text,object
label,int64


In [11]:
dict(a)

{0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'}

In [12]:
# Convert 'label' column to integer type
df['label'] = df['label'].astype(int)

# Now map
a = zip([0, 1, 2, 3, 4, 5], ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise'])
df['mapped_label'] = df['label'].map(dict(a))

In [13]:
df.groupby(['mapped_label']).count()

,text,label
mapped_label,,
anger,2159,2159
fear,1937,1937
joy,5362,5362
love,1304,1304
sadness,4666,4666
surprise,572,572


## Tokenization of Data from the Raw String

In [14]:
from transformers import AutoTokenizer

model_checkpoint= "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

text = "My name is Riddhiman Sherlekar"
encoded_text = tokenizer(text)
print(encoded_text)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

{'input_ids': [101, 2026, 2171, 2003, 9436, 19114, 2386, 2016, 20927, 6673, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


## Split the Data as :

1) Training Data
2) Test Data
3) Validation Data

In [15]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, test_size = 0.3, stratify=df['mapped_label'])
test, validation = train_test_split(test, test_size = 1/3, stratify = test['label'])


train.shape , test.shape , validation.shape, df.shape

((11200, 3), (3200, 3), (1600, 3), (16000, 3))

In [16]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})

In [17]:
from datasets import Dataset, DatasetDict

dataset = DatasetDict({
    'train' : Dataset.from_pandas(train, preserve_index = False),
    'test' : Dataset.from_pandas(test, preserve_index = False),
    'validation' : Dataset.from_pandas(validation, preserve_index = False)
})

dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'mapped_label'],
        num_rows: 11200
    })
    test: Dataset({
        features: ['text', 'label', 'mapped_label'],
        num_rows: 3200
    })
    validation: Dataset({
        features: ['text', 'label', 'mapped_label'],
        num_rows: 1600
    })
})

In [18]:
dataset['train'][0] , dataset['test'][0] , dataset['validation'][0]

({'text': 'i am mostly feeling contentedly terrified about it all',
  'label': 4,
  'mapped_label': 'fear'},
 {'text': 'i fall off when my uncle hits so i cant imagine what it must feel like to go mph other than cool and possibly painful',
  'label': 1,
  'mapped_label': 'joy'},
 {'text': 'i swear it made me feel a lot better',
  'label': 1,
  'mapped_label': 'joy'})

Tokenize all the text in Dataset using the Map function of HuggingFace

In [19]:
def tokenize(batch):
  return tokenizer(batch['text'], padding= True, truncation=True)

tokenize(dataset['train'][:3])

{'input_ids': [[101, 1045, 2572, 3262, 3110, 4180, 26207, 10215, 2055, 2009, 2035, 102, 0, 0, 0], [101, 1045, 2572, 3110, 2074, 2061, 7653, 2157, 2085, 102, 0, 0, 0, 0, 0], [101, 1045, 6135, 1998, 3294, 2514, 2489, 2725, 2008, 2003, 5921, 2066, 13128, 9293, 102]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}

In [20]:
encoded = dataset.map(tokenize, batched=True, batch_size = None)

Map:   0%|          | 0/11200 [00:00<?, ? examples/s]

Map:   0%|          | 0/3200 [00:00<?, ? examples/s]

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

In [21]:
encoded

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'mapped_label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 11200
    })
    test: Dataset({
        features: ['text', 'label', 'mapped_label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3200
    })
    validation: Dataset({
        features: ['text', 'label', 'mapped_label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1600
    })
})

In [22]:
label_names['label']

ClassLabel(names=['sadness', 'joy', 'love', 'anger', 'fear', 'surprise'], id=None)

In [23]:
label_names = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']

Working with models

In [24]:
from transformers import AutoModel
model = AutoModel.from_pretrained(model_checkpoint)

model.config.id2label, model.config.label2id

label2id = {label: i for i , label in enumerate(label_names)}
id2label = {i: label for i, label in enumerate(label_names)}


label2id, id2label

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

({'sadness': 0, 'joy': 1, 'love': 2, 'anger': 3, 'fear': 4, 'surprise': 5},
 {0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'})

In [25]:
model

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [26]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, AutoConfig
import torch

if torch.has_mps:
  device = torch.device("mps")
elif torch.cuda.is_available():
  device = torch.device("cuda")
else:
  device = torch.device("cpu")


print(device)

config = AutoConfig.from_pretrained(model_checkpoint, label2id = label2id , id2label = id2label)
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, config=config).to(device)




/tmp/ipython-input-26-1548317624.py:4: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  if torch.has_mps:


cuda


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [27]:
#!pip install accelerate

!pip show accelerate

Name: accelerate
Version: 1.8.1
Summary: Accelerate
Home-page: https://github.com/huggingface/accelerate
Author: The HuggingFace team
Author-email: zach.mueller@huggingface.co
License: Apache
Location: /usr/local/lib/python3.11/dist-packages
Requires: huggingface_hub, numpy, packaging, psutil, pyyaml, safetensors, torch
Required-by: peft


In [28]:
from transformers import TrainingArguments
batch_size = 64
training_dir = "bert_base_uncased_trained_model"


training_args = TrainingArguments(output_dir=training_dir, overwrite_output_dir=True,
                                  num_train_epochs=2,
                                  learning_rate= 2e-5,
                                  per_device_eval_batch_size = batch_size,
                                  per_device_train_batch_size = batch_size,
                                  weight_decay = 0.01,
                                  eval_strategy='epoch',
                                  disable_tqdm=False)

In [29]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(pred):
  labels = pred.label_ids
  preds = pred.predictions.argmax(-1)

  f1 = f1_score(labels, preds , average="weighted")
  acc = accuracy_score(labels, preds)

  return {"accuracy" : acc, "F1 score": f1}

 Build the Compute Metrics

In [30]:
from transformers import Trainer

trainer = Trainer(
    model = model ,
    args = training_args,
    compute_metrics = compute_metrics,
    train_dataset = encoded['train'],
    eval_dataset = encoded['validation'],
    tokenizer = tokenizer
)

/tmp/ipython-input-30-501311670.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [31]:
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: riddhisher (riddhisher-cisco) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Accuracy,F1 score
1,No log,0.535442,0.839375,0.828843
2,No log,0.315171,0.903750,0.903489


TrainOutput(global_step=350, training_loss=0.7305397687639509, metrics={'train_runtime': 339.1987, 'train_samples_per_second': 66.038, 'train_steps_per_second': 1.032, 'total_flos': 1001502421516800.0, 'train_loss': 0.7305397687639509, 'epoch': 2.0})

Save the model

In [32]:
trainer.save_model("bert_base_uncased_text-classification-model")

In [33]:
from transformers import pipeline

classifier = pipeline('text-classification' , model="bert_base_uncased_text-classification-model")


classifier([
    'I like ML',
    'I started annoyed with laptops',
    'I am low today',
    'I am tensed if I am not going to get the job in ML',
    'I am worried with the current job market'





])

Device set to use cuda:0


[{'label': 'love', 'score': 0.5661706328392029},
 {'label': 'anger', 'score': 0.9201338291168213},
 {'label': 'sadness', 'score': 0.9398444294929504},
 {'label': 'fear', 'score': 0.8135121464729309},
 {'label': 'fear', 'score': 0.9055148363113403}]

In [34]:
from transformers import pipeline, AutoModel, AutoTokenizer

# Load your pipeline
classifier = pipeline('text-classification', model="bert_base_uncased_text-classification-model")

# Extract the model and tokenizer
model = classifier.model
tokenizer = classifier.tokenizer


# Push model and tokenizer
model.push_to_hub("rsher60/bert_base_uncased_text-classification-model")
tokenizer.push_to_hub("rsher60/bert_base_uncased_text-classification-model")

Device set to use cuda:0


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/rsher60/bert_base_uncased_text-classification-model/commit/5fdd9cf3de37f1b61303387a2c9f4c121ca020f8', commit_message='Upload tokenizer', commit_description='', oid='5fdd9cf3de37f1b61303387a2c9f4c121ca020f8', pr_url=None, repo_url=RepoUrl('https://huggingface.co/rsher60/bert_base_uncased_text-classification-model', endpoint='https://huggingface.co', repo_type='model', repo_id='rsher60/bert_base_uncased_text-classification-model'), pr_revision=None, pr_num=None)

## Inference the model from HF

In [ ]:
from transformers import pipeline

classifier =